# SETUP FOR BETTER GPU

1) Install UCSD VPN: https://blink.ucsd.edu/technology/network/connections/off-campus/VPN/
2) ssh on local machine to ssh username@dsmlp-login.ucsd.edu where username is <username>@ucsd.edu
3) Run the following command on the ssh: launch.sh -c 8 -m 16 -g 1 -v a30
4) Open the link it provides
5) Make sure to **shut down** the pod when you're done from File > Shut Down. And then exit ssh from your terminal

6) If it says GPU already in use (1 of 1 allocated): run the following "kubectl get pods" and "kubectl delete pod <paste-pod-name-here>"


## Starter Code for Datahub

### IF THIS IS YOUR FIRST TIME RUNNING THIS ON DATA HUB
You need to run the cell immediately below so that you can install/setup the kernel. Its called "Python (cse151b)". You should **always** select this kernel in the top right of jupyter. 

### If you're coming back, make sure you re-run everything from the next cell (imports) and further down

The purpose of this starter code is so that you can basically duplicate this and run your experiments in a separate notebook.

# START HERE IF ITS YOUR FIRST TIME HERE

In [1]:
# =========================================================
# DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# =========================================================

# remove old kernel
!jupyter kernelspec uninstall -y cse151b

# Remove old .venv
!rm -rf .venv

# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

print("Old environment and kernel destroyed.")

# Create a virtual environment
!~/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter scikit-learn pandas

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

Couldn't find kernel spec(s): cse151b
downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /home/a5verma/.local/bin
  uv
  uvx
everything's installed!
Old environment and kernel destroyed.
Using CPython 3.11.9 interpreter at: /opt/conda/bin/python
Creating virtual environment with seed packages at: .venv
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.2
 + pip==26.1
 + setuptools==82.0.1
 + wheel==0.47.0
Activate with: source .venv/bin/activate
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433

# START HERE IF YOU'VE ALREADY SETUP! 

# MAKE SURE TO RESTART KERNEL BEFORE CONTINUING!

## Library Setup

In [2]:
!source ./.venv/bin/activate

import json
import os
import random

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm
from datetime import datetime 
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

## Config + Data Loading
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [145]:
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0" 
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/datahub_experiments.jsonl"
MAX_TOKENS  = 16384

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## Hyperparam Sandbox + LLM Setup

If this doesn't work, what worked for me was:
1) shut down ALL OTHER kernels in the directory
2) re-run the first cell (commented out one)

If it still fails:
1) run "pkill -9 -f python" in terminal. This will kill the datahub pod and (most likely) wipe /dev/shm
2) re-launch the container
3) ensure all other kernels are shut down and go from the top.

In [4]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     enforce_eager=True,
#     gpu_memory_utilization=0.85,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model and parameters loaded successfully in Eager Mode.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16", 
    quantization=None, 
    load_format="auto",
    enable_prefix_caching=True, 
    enforce_eager=False,        
    gpu_memory_utilization=0.85, 
    max_model_len=4096, # 16384
    trust_remote_code=True,
    max_num_seqs=32, # 128
    max_num_batched_tokens=4096, # 16384
)

print("Model loaded with CUDAGraph + BF16.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=1217) INFO 05-01 22:58:52 [default_loader.py:384] Loading weights took 1.36 seconds
(EngineCore pid=1217) INFO 05-01 22:58:53 [gpu_model_runner.py:4820] Model loading took 7.61 GiB memory and 10.467743 seconds
(EngineCore pid=1217) INFO 05-01 22:59:06 [backends.py:1051] Using cache directory: /tmp/xdg-cache/vllm/torch_compile_cache/eaaa5d6a7a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1217) INFO 05-01 22:59:06 [backends.py:1111] Dynamo bytecode transform time: 12.76 s


(EngineCore pid=1217) [rank0]:W0501 22:59:08.132000 1217 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=1217) INFO 05-01 22:59:13 [backends.py:372] Cache the graph of compile range (1, 4096) for later use
(EngineCore pid=1217) INFO 05-01 22:59:19 [backends.py:390] Compiling a graph for compile range (1, 4096) takes 12.12 s
(EngineCore pid=1217) INFO 05-01 22:59:22 [decorators.py:655] saved AOT compiled function to /tmp/xdg-cache/vllm/torch_compile_cache/torch_aot_compile/213c05d26476ced6432f65e580752dbb6b56b6ec097cf1fbdc507d79f5857e4b/rank_0_0/model
(EngineCore pid=1217) INFO 05-01 22:59:22 [monitor.py:48] torch.compile took 28.28 s in total
(EngineCore pid=1217) INFO 05-01 22:59:23 [monitor.py:76] Initial profiling/warmup run took 1.39 s
(EngineCore pid=1217) INFO 05-01 22:59:30 [kv_cache_utils.py:829] Overriding num_gpu_blocks=0 with num_gpu_blocks_override=64
(EngineCore pid=1217) INFO 05-01 22:59:30 [gpu_model_runner.py:5876] Profiling CUDA graph memory: PIECEWISE=11 (largest=64), FULL=7 (largest=32)
(EngineCore pid=1217) INFO 05-01 22:59:31 [gpu_model_runner.py:5955]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 11/11 [00:00<00:00, 21.73it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 7/7 [00:00<00:00, 24.89it/s]


(EngineCore pid=1217) INFO 05-01 22:59:34 [gpu_model_runner.py:6046] Graph capturing finished in 2 secs, took 0.46 GiB
(EngineCore pid=1217) INFO 05-01 22:59:34 [gpu_worker.py:597] CUDA graph pool memory: 0.46 GiB (actual), 0.51 GiB (estimated), difference: 0.05 GiB (10.6%).
(EngineCore pid=1217) INFO 05-01 22:59:34 [core.py:283] init engine (profile, create kv cache, warmup model) took 40.51 seconds
Model loaded with CUDAGraph + BF16.


In [146]:
NUM_SEQS = 5
sampling_params = SamplingParams(
    n=NUM_SEQS,
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    presence_penalty=0.0,
    repetition_penalty=1.05, # slight penalty for repetition
)

## Experiment/Submission Config - CONFIGURE ME!

In [147]:
IS_EXPERIMENT   = True                        # Set to False to run the full 1126 dataset
EXPERIMENT_NAME = "self_consistency_experiment" # Only used if IS_EXPERIMENT is True
SAVE_EVAL       = True                        # Set to False for private test set (no ground truth)

# Dynamic Naming Logic
date_str = datetime.now().strftime("%Y-%m-%d")

if IS_EXPERIMENT:
    output_filename = f"experiment_{EXPERIMENT_NAME}_{date_str}.jsonl"
else:
    output_filename = f"submission_{date_str}.jsonl"

OUTPUT_PATH = Path("results") / output_filename
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"RUN MODE : {'EXPERIMENT (50 samples)' if IS_EXPERIMENT else 'FULL DATASET (1126 items)'}")
print(f"SAVE TO  : {OUTPUT_PATH}")

RUN MODE : EXPERIMENT (50 samples)
SAVE TO  : results/experiment_self_consistency_experiment_2026-05-02.jsonl


## Prompt Engineering Sandbox

In [148]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

## Dataset Generation

In [149]:
if IS_EXPERIMENT:
    SUBSET_SIZE = 50
    RANDOM_SEED = 42
    random.seed(RANDOM_SEED)

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]

    mcq_ratio = len(mcq_pool) / len(data)
    n_mcq  = max(1, round(SUBSET_SIZE * mcq_ratio))
    n_free = SUBSET_SIZE - n_mcq

    data_to_run = random.sample(mcq_pool, n_mcq) + random.sample(free_pool, n_free)
    random.shuffle(data_to_run)
else:
    data_to_run = data

print(f"Data ready. Queuing {len(data_to_run)} questions for inference.")

Data ready. Queuing 50 questions for inference.


## Inference 

In [150]:
prompts = []
for item in data_to_run:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions...")

# Chunking to prevent EngineDead errors
BATCH_SIZE = 1
all_outputs = []

for i in range(0, len(prompts), BATCH_SIZE):
    batch = prompts[i:i + BATCH_SIZE]
    print(f"Processing batch {i//BATCH_SIZE + 1} / {(len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE}")
    outputs = llm.generate(batch, sampling_params=sampling_params)
    all_outputs.extend(outputs)

parallel_responses = [[out.outputs[i].text.strip() for i in range(NUM_SEQS)] for out in all_outputs]
# responses = [response[0] for response in parallel_responses]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data_to_run[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 50 questions...
Processing batch 1 / 50


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 2 / 50


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 50 / 50


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=910) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so we have triangle HGU with centroid B and incenter X. Angles at H and G are N and J, so let's clarify the triangle labeling first to avoid confusion. Wait, vertices: H, G, U? So sides opposite: wait, maybe better to fix notation properly. Let's say in triangle HG ...

── Response 1 (id=308) ──
Okay, let's try to figure out this problem step by step. First, I need to remember what the previous problem was about because the user mentioned "the formula derived in the previous problem." Since I don't have the previous problem here, I might need to make some assumptions based on common problems like this. Usually, when a real estate agent says the land costs more than the house, and they men ...

── Response 2 (id=54) ──
Okay, let's see. I need to find the period and amplitude of the function \( r = 0.7 \sin(6t) + 6 \). Hmm, firs

## Score Responses + Summary

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [44]:
for item, response in tqdm(zip(data_to_run, parallel_responses), total=len(data_to_run), desc="Scoring"):
    print(item)
    print(response)

Scoring: 100%|██████████| 3/3 [00:00<00:00, 2933.08it/s]

{'question': 'Consider a triangle $\\triangle HGU$. The centroid is denoted by $B$, while $X$ represents the incenter. The angles at vertices $H$ and $G$ are labeled as $N$ and $J$ respectively. Given that the line segment $XB$ runs parallel to side $HG$, and $J$ is equal to $2 \\arctan (1/3)$, determine the value of $N$.', 'options': ['\\frac{\\pi}{4}', '\\frac{\\pi}{3}', '\\frac{3\\pi}{8}', '\\frac{3\\pi}{4}', '\\frac{\\pi}{6}', '\\frac{\\pi}{5}', '\\frac{\\pi}{2}'], 'answer': 'G', 'id': 974}
["This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.\nWell, so we have triangle HGU with centroid B and incenter X. Angles at H and G are N and J, so let's clarify the triangle labeling first to avoid confusion. Wait, vertices: H, G, U? So sides opposite: wait, maybe better to fix notation properly. Let's say in triangle HGU, vertices are H, G, U, so angle at H is ∠H = N, angle at G is ∠G = J, so then angle at U must b

In [151]:
from collections import Counter 

def get_mode(votes):
    n = len(votes) 
    
    data = Counter(votes) 
    get_mode = dict(data) 
    mode = [k for k, v in get_mode.items() if v == max(list(data.values()))] 
    
    return mode[0]

# Load Judger
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m: return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

In [164]:
responses = [response[0] for response in parallel_responses]
results = []
for item, response in tqdm(zip(data_to_run, responses), total=len(data_to_run), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = extract_letter(response) == str(gold).strip().upper()
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
        except Exception:
            correct = False

    results.append({
        "id": item.get("id"), "is_mcq": is_mcq, "gold": gold,
        "response": response, "correct": correct,
    })

Scoring: 100%|██████████| 50/50 [00:01<00:00, 31.11it/s]


In [166]:
from collections import Counter 
import re

def extract_free_response(text: str) -> str:
    """Extracts the content inside \boxed{}."""
    m = re.search(r"\\boxed\{(.*?)\}", text)
    if m:
        return f"\\boxed{{{m.group(1).strip()}}}"
    
    return text.strip()

def get_mode(votes):
    n = len(votes) 
    
    data = Counter(votes) 
    get_mode = dict(data) 
    mode = [k for k, v in get_mode.items() if v == max(list(data.values()))] 
    
    return mode[0]
    
results = []
for item, response in tqdm(zip(data_to_run, parallel_responses), total=len(data_to_run), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        pred_letter = get_mode([extract_letter(response[i]) for i in range(NUM_SEQS)])
        correct = pred_letter == str(gold).strip().upper()
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        
        pred_answer = get_mode([extract_free_response(response[i]) for i in range(NUM_SEQS)])
        final_pred = pred_answer
        
        try:
            correct = judger.auto_judge(pred=response[0], gold=gold_list, options=[[]] * len(gold_list))
        except Exception:
            correct = False

    results.append({
        "id": item.get("id"), "is_mcq": is_mcq, "gold": gold,
        "response": response, "correct": correct,
    })

Scoring: 100%|██████████| 50/50 [00:01<00:00, 30.40it/s]


In [168]:
# --- PANDAS METRICS TABLE ---
def generate_metrics_df(results):
    rows = []
    for subset_name, is_mcq_flag in [("Overall", None), ("MCQ", True), ("Free-Form", False)]:
        if is_mcq_flag is None:
            subset = results
        else:
            subset = [r for r in results if r["is_mcq"] == is_mcq_flag]
            
        if not subset: continue
            
        y_true = [1] * len(subset)
        y_pred = [1 if r["correct"] else 0 for r in subset]
        
        rows.append({
            "Category": subset_name,
            "Accuracy": f"{accuracy_score(y_true, y_pred):.2%}",
            "F1 Score": round(f1_score(y_true, y_pred, zero_division=0), 3),
            "Correct": sum(y_pred),
            "Total": len(y_pred)
        })
    return pd.DataFrame(rows).set_index("Category")

metrics_df = generate_metrics_df(results)
display(metrics_df)

,Accuracy,F1 Score,Correct,Total
Category,,,,
Overall,38.00%,0.551,19,50
MCQ,23.53%,0.381,4,17
Free-Form,45.45%,0.625,15,33


## Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [10]:
with open(OUTPUT_PATH, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {OUTPUT_PATH}")

Saved 50 records to results/experiment_no_few_shot_baseline_2026-05-01.jsonl
